In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What is the best way to reduce fatigue in Long COVID?"

## Abstract

Among 595 users on r/covidlonghaulers who discussed fatigue between March 11 and April 10, 2026, community treatment reports reveal a clear hierarchy of patient-reported benefit. Magnesium (93% user-level positive rate, n=29), B vitamins (94%, n=16), quercetin (93%, n=14), and vitamin D (82%, n=38) lead the field among supplements. Low-dose naltrexone (LDN) is the most-discussed prescription treatment with a 73% positive rate across 94 fatigue-reporting users. SSRIs are the most divisive treatment class, with a 44% positive rate and the only negative mean sentiment among widely-used options. The analysis applies user-level aggregation with Wilson score confidence intervals, Fisher's exact tests, and effect size metrics to 381 unique fatigue-reporting treatment users across 48 treatments meeting the minimum sample threshold.

## Data Exploration

Data covers: 2026-03-11 to 2026-04-10 (1 month). Source: r/covidlonghaulers via PatientPunk extraction pipeline.

In [ ]:

# Identify fatigue-mentioning users
fatigue_query = """
SELECT DISTINCT user_id FROM posts
WHERE body_text LIKE '%fatigue%'
OR body_text LIKE '%tired%'
OR body_text LIKE '%exhausted%'
OR body_text LIKE '%exhaustion%'
OR body_text LIKE '%low energy%'
OR body_text LIKE '%energy level%'
"""
fatigue_users = pd.read_sql(fatigue_query, conn)
fatigue_ids = set(fatigue_users['user_id'])

# All treatment reports (user-level aggregation)
all_reports = pd.read_sql("""
SELECT tr.user_id, t.canonical_name,
    tr.sentiment, tr.signal_strength
FROM treatment_reports tr
JOIN treatment t ON tr.drug_id = t.id
WHERE t.canonical_name NOT IN ('supplements','medication','treatment','therapy','drug','drugs',
    'vitamin','prescription','pill','pills','dosage','dose')
""", conn)

all_reports['is_fatigue'] = all_reports['user_id'].isin(fatigue_ids)
all_reports['sent_score'] = all_reports['sentiment'].map(SENTIMENT_SCORE).fillna(0.0)

# User-level aggregation
user_drug = all_reports.groupby(['user_id', 'canonical_name', 'is_fatigue']).agg(
    avg_sent=('sent_score', 'mean'),
    n_reports=('sent_score', 'count'),
    max_strength=('signal_strength', lambda x: 'strong' if 'strong' in x.values else ('moderate' if 'moderate' in x.values else 'weak'))
).reset_index()
user_drug['outcome'] = user_drug['avg_sent'].apply(classify_outcome)

fatigue_drug = user_drug[user_drug['is_fatigue']].copy()

# Causal context exclusions
CAUSAL_TERMS = {'covid vaccine', 'vaccine', 'flu vaccine', 'mmr vaccine', 'moderna vaccine',
    'mrna covid-19 vaccine', 'pfizer vaccine', 'vaccine injection', 'pfizer', 'booster'}
fatigue_drug_clean = fatigue_drug[~fatigue_drug['canonical_name'].isin(CAUSAL_TERMS)].copy()

# Merge duplicates
MERGE_MAP = {'pepcid': 'famotidine', 'magnesium glycinate': 'magnesium'}
fatigue_drug_clean['canonical_name'] = fatigue_drug_clean['canonical_name'].replace(MERGE_MAP)

# Re-aggregate after merges
fatigue_drug_clean = fatigue_drug_clean.groupby(['user_id', 'canonical_name']).agg(
    avg_sent=('avg_sent', 'mean'),
    n_reports=('n_reports', 'sum'),
).reset_index()
fatigue_drug_clean['outcome'] = fatigue_drug_clean['avg_sent'].apply(classify_outcome)

# Drug summary
drug_summary = fatigue_drug_clean.groupby('canonical_name').agg(
    n_users=('user_id', 'nunique'),
    pos_rate=('outcome', lambda x: (x == 'positive').mean()),
    neg_rate=('outcome', lambda x: (x == 'negative').mean()),
    mixed_rate=('outcome', lambda x: (x == 'mixed/neutral').mean()),
    mean_sent=('avg_sent', 'mean'),
).reset_index()

drug_summary['pos_count'] = drug_summary.apply(
    lambda r: int(round(r['pos_rate'] * r['n_users'])), axis=1)
drug_summary[['ci_low', 'ci_high']] = drug_summary.apply(
    lambda r: pd.Series(wilson_ci(r['pos_count'], r['n_users'])), axis=1)

drug_main = drug_summary[drug_summary['n_users'] >= 10].sort_values('pos_rate', ascending=False).copy()

# Display key stats
stats_html = f"""
<table style="font-size: 14px; border-collapse: collapse; width: 60%;">
<tr><td style="padding: 6px; border-bottom: 1px solid #ddd;"><b>Total users in dataset</b></td><td style="padding: 6px; border-bottom: 1px solid #ddd;">2,827</td></tr>
<tr><td style="padding: 6px; border-bottom: 1px solid #ddd;"><b>Users mentioning fatigue</b></td><td style="padding: 6px; border-bottom: 1px solid #ddd;">{len(fatigue_ids)} (21% of community)</td></tr>
<tr><td style="padding: 6px; border-bottom: 1px solid #ddd;"><b>Fatigue users with treatment reports</b></td><td style="padding: 6px; border-bottom: 1px solid #ddd;">{fatigue_drug_clean['user_id'].nunique()}</td></tr>
<tr><td style="padding: 6px; border-bottom: 1px solid #ddd;"><b>Treatments meeting threshold (n&ge;10)</b></td><td style="padding: 6px; border-bottom: 1px solid #ddd;">{len(drug_main)}</td></tr>
<tr><td style="padding: 6px; border-bottom: 1px solid #ddd;"><b>Overall positive rate (fatigue users)</b></td><td style="padding: 6px; border-bottom: 1px solid #ddd;">{fatigue_drug_clean['outcome'].eq('positive').mean():.1%}</td></tr>
<tr><td style="padding: 6px;"><b>Causal-context exclusions</b></td><td style="padding: 6px;">Vaccines (covid vaccine, pfizer, moderna, booster, etc.) excluded &mdash; negative sentiment reflects perceived cause of illness, not treatment response</td></tr>
</table>
"""
display(HTML(stats_html))


Pepcid and famotidine were merged (same compound, 11 users reported both). Magnesium glycinate reports were merged into the magnesium category (10 users reported both). Generic terms (supplements, medication, vitamin, etc.) were excluded.

## The Fatigue Treatment Landscape

Before examining which treatments perform best, we need the lay of the land: what are fatigue-reporting Long COVID patients actually trying, and what does the overall distribution of outcomes look like?

In [ ]:

# Top 25 treatments diverging bar chart
top25 = drug_main.head(25).sort_values('n_users', ascending=True).copy()

fig, ax = plt.subplots(figsize=(13, 9))
y = range(len(top25))
labels = top25['canonical_name'].values
pos_pct = top25['pos_rate'].values * 100
neg_pct = top25['neg_rate'].values * 100
mix_pct = top25['mixed_rate'].values * 100

ax.barh(y, -mix_pct, left=0, color='#95a5a6', label='Mixed/Neutral', height=0.7)
ax.barh(y, -neg_pct, left=-mix_pct, color='#e74c3c', label='Negative', height=0.7)
ax.barh(y, pos_pct, left=0, color='#2ecc71', label='Positive', height=0.7)

for i, (n, pr) in enumerate(zip(top25['n_users'].values, pos_pct)):
    ax.text(pr + 2, i, f'n={n}', va='center', fontsize=8, color='#555')

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Percentage of Users (%)', fontsize=11)
ax.set_title('Treatment Outcomes Among Fatigue-Reporting Long COVID Users\n(User-level aggregation, n\u226510)', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=10)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


The diverging bar chart reveals a wide spread in treatment outcomes. Magnesium, B vitamins, quercetin, and B12 cluster at the top with positive rates above 85%. At the bottom, SSRIs (44%), antibiotics (47%), and cromolyn sodium (40%) are the only treatments where fewer than half of users report positive outcomes. Most treatments fall in a middle tier between 60-80% positive. The community overall positive rate for fatigue users is approximately 71%, which serves as our baseline for distinguishing outperformers from underperformers.

## Statistical Ranking: Which Treatments Beat Chance?

A raw positive rate is misleading when sample sizes vary from 10 to 94. A treatment with 10/10 positive reports looks perfect, but the uncertainty is enormous. We use a binomial test against a 50% null (do more users report benefit than not?) combined with Wilson score confidence intervals to identify treatments with statistically reliable positive performance.

In [ ]:

import math
from scipy.stats import binomtest

results = []
for _, row in drug_main.iterrows():
    n = int(row['n_users'])
    k = int(row['pos_count'])
    bt = binomtest(k, n, 0.5, alternative='greater')
    p_obs = k / n if n > 0 else 0
    h = 2 * (math.asin(math.sqrt(p_obs)) - math.asin(math.sqrt(0.5)))
    nnt_val = nnt(p_obs, 0.5) if p_obs > 0.5 else None
    results.append({
        'Treatment': row['canonical_name'],
        'Users (n)': n,
        'Positive Rate': f"{row['pos_rate']:.0%}",
        'Wilson 95% CI': f"[{row['ci_low']:.0%}, {row['ci_high']:.0%}]",
        'p-value': bt.pvalue,
        'Cohen h': round(h, 2),
        'NNT': nnt_val,
        'pos_rate_raw': row['pos_rate'],
    })

results_df = pd.DataFrame(results).sort_values('pos_rate_raw', ascending=False)

def sig_marker(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.10: return '\u2020'
    return 'ns'

results_df['Sig'] = results_df['p-value'].apply(sig_marker)
results_df['p-value'] = results_df['p-value'].apply(lambda x: f"{x:.4f}" if x >= 0.001 else f"{x:.1e}")
results_df['NNT'] = results_df['NNT'].apply(lambda x: f"{x:.1f}" if x else '\u2014')

display_df = results_df[['Treatment', 'Users (n)', 'Positive Rate', 'Wilson 95% CI', 'p-value', 'Sig', 'Cohen h', 'NNT']].head(30)
styled = display_df.style.set_caption(
    "Binomial test: H\u2080 = 50% positive rate (coin flip). *** p<0.001, ** p<0.01, * p<0.05, \u2020 p<0.10, ns = not significant"
).hide(axis='index').set_properties(**{'font-size': '11px', 'text-align': 'center'}).set_properties(
    subset=['Treatment'], **{'text-align': 'left'})
display(styled)


Twelve treatments achieve statistical significance (p<0.05) against a 50% baseline, meaning we can be confident that more users report benefit than not. Magnesium (93%, p<0.001, Cohen's h=1.15), B vitamins (94%, p<0.001), and quercetin (93%, p=0.001) show the largest effect sizes, though all have sample sizes under 30. LDN stands out as the only treatment combining a large sample (n=94), statistical significance (p<0.001), and a meaningful effect size (h=0.48). SSRIs are the only widely-used treatment where we cannot reject the null hypothesis that outcomes are no better than chance (44% positive, p=0.69).

The NNT (number needed to treat) column translates these numbers into patient-level decisions. For LDN, an NNT of 4.3 means roughly 1 in 4 people who try LDN can expect to report a positive outcome beyond what chance alone would predict. For magnesium (NNT=2.3), it is roughly 1 in 2.

In [ ]:

import math
top20 = drug_main.head(20).sort_values('pos_rate', ascending=True).copy()

fig, ax = plt.subplots(figsize=(11, 8))
y_pos = range(len(top20))

colors_forest = []
for _, r in top20.iterrows():
    p = binomtest(int(round(r['pos_rate']*r['n_users'])), int(r['n_users']), 0.5, alternative='greater').pvalue
    colors_forest.append('#2ecc71' if p < 0.05 else '#95a5a6')

ax.scatter(top20['pos_rate']*100, y_pos, c=colors_forest, s=top20['n_users']*3, zorder=5, edgecolors='white', linewidth=0.5)

for i, (_, row) in enumerate(top20.iterrows()):
    ax.plot([row['ci_low']*100, row['ci_high']*100], [i, i], color=colors_forest[i], linewidth=2, zorder=4)

ax.axvline(50, color='#e74c3c', linestyle='--', linewidth=1.2, label='50% null (coin flip)', zorder=3)
baseline_rate = fatigue_drug_clean['outcome'].eq('positive').mean() * 100
ax.axvline(baseline_rate, color='#3498db', linestyle=':', linewidth=1.2, label=f'Community baseline ({baseline_rate:.0f}%)', zorder=3)

ax.set_yticks(y_pos)
ax.set_yticklabels(top20['canonical_name'].values, fontsize=9)
ax.set_xlabel('User-Level Positive Rate (%)', fontsize=11)
ax.set_title('Treatment Effectiveness for Fatigue: Positive Rate with 95% Wilson CIs\n(Dot size = sample size; green = p<0.05)', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(10, 105)
fig.tight_layout()
plt.show()


The forest plot makes the uncertainty visible. Magnesium and quercetin sit at the far right with confidence intervals that clear the 50% line by a wide margin, but their dots are small (n<30). LDN's dot is large (n=94) with a CI that comfortably excludes 50%. Meanwhile, SSRIs' confidence interval straddles the 50% line, confirming the binomial test result: we cannot conclude that SSRIs produce more positive than negative outcomes for fatigue in this population.

## Head-to-Head: Do Top Treatments Outperform Each Other?

Comparing each treatment to a 50% baseline tells us which ones "work" in isolation, but patients want to know which is better. We compare the top performer by sample size (LDN, n=94) against key alternatives using Fisher's exact test on user-level outcomes.

In [ ]:

import math
ldn_users_df = fatigue_drug_clean[fatigue_drug_clean['canonical_name'] == 'low dose naltrexone']
ldn_pos = (ldn_users_df['outcome'] == 'positive').sum()
ldn_neg = len(ldn_users_df) - ldn_pos

comparisons = ['magnesium', 'vitamin d', 'coq10', 'nicotine', 'nattokinase',
               'creatine', 'ssri', 'electrolyte', 'probiotics']

comp_results = []
for drug in comparisons:
    drug_users_df = fatigue_drug_clean[fatigue_drug_clean['canonical_name'] == drug]
    n = len(drug_users_df)
    if n < 10:
        continue
    d_pos = (drug_users_df['outcome'] == 'positive').sum()
    d_neg = n - d_pos
    table = [[ldn_pos, ldn_neg], [d_pos, d_neg]]
    odds_ratio, p_val = fisher_exact(table)
    p1 = ldn_pos / len(ldn_users_df)
    p2 = d_pos / n
    h = 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))
    comp_results.append({
        'Comparison': f'LDN vs {drug}',
        'LDN (n)': f"{ldn_pos}/{len(ldn_users_df)} ({p1:.0%})",
        'Comparator (n)': f"{d_pos}/{n} ({p2:.0%})",
        'Odds Ratio': round(odds_ratio, 2),
        'p-value': p_val,
        'Cohen h': round(h, 2),
        'Direction': 'LDN better' if p1 > p2 else ('Equal' if abs(p1-p2)<0.01 else 'Comparator better'),
    })

comp_df = pd.DataFrame(comp_results)
comp_df['Sig'] = comp_df['p-value'].apply(sig_marker)
comp_df['p-value'] = comp_df['p-value'].apply(lambda x: f"{x:.3f}")
styled_comp = comp_df.style.set_caption(
    "Fisher's exact test comparing LDN positive rate to alternatives among fatigue users"
).hide(axis='index').set_properties(**{'font-size': '11px', 'text-align': 'center'}).set_properties(
    subset=['Comparison'], **{'text-align': 'left'})
display(styled_comp)


LDN does not significantly outperform any of the top supplements (magnesium, vitamin D, CoQ10, nattokinase, creatine) in pairwise comparisons. This is partly a power issue (most comparators have n<35), but also because several supplements genuinely show comparable or higher positive rates. The one statistically significant comparison is LDN vs SSRIs, where LDN clearly outperforms. Magnesium's higher point estimate (93% vs 73%) is notable but the small sample (n=29) means we cannot confirm the difference is reliable.

For patients, the practical takeaway: the top supplements and LDN are in the same performance tier. SSRIs are in a lower tier.

In [ ]:

CATEGORIES = {
    'Supplements': ['magnesium', 'b vitamins', 'quercetin', 'b12', 'vitamin d', 'vitamin c',
                    'coq10', 'omega-3', 'taurine', 'n-acetylcysteine', 'creatine', 'electrolyte'],
    'Antihistamine/Mast Cell': ['antihistamines', 'cetirizine', 'fexofenadine', 'ketotifen',
                                 'h1 antihistamine', 'h2 antihistamine', 'famotidine', 'cromolyn sodium'],
    'Prescription (other)': ['low dose naltrexone', 'propranolol', 'beta blocker', 'guanfacine',
                             'ivabradine', 'fluvoxamine', 'steroids', 'paxlovid', 'glp-1 receptor agonist'],
    'SSRIs/Antidepressants': ['ssri', 'antidepressants'],
    'Alternative/Novel': ['nicotine', 'nattokinase', 'red light therapy', 'methylene blue',
                          'peptide', 'nad', 'vagus nerve stimulation'],
}

cat_data = []
for cat, drugs in CATEGORIES.items():
    cat_users = fatigue_drug_clean[fatigue_drug_clean['canonical_name'].isin(drugs)]
    n = cat_users['user_id'].nunique()
    if n < 5:
        continue
    pos = (cat_users['outcome'] == 'positive').sum()
    neg = (cat_users['outcome'] == 'negative').sum()
    mix = (cat_users['outcome'] == 'mixed/neutral').sum()
    total = len(cat_users)
    ci_l, ci_h = wilson_ci(pos, total)
    cat_data.append({
        'Category': cat, 'n_users': n,
        'pos_rate': pos/total if total else 0,
        'neg_rate': neg/total if total else 0,
        'mix_rate': mix/total if total else 0,
        'ci_low': ci_l, 'ci_high': ci_h, 'total': total,
    })

cat_df = pd.DataFrame(cat_data).sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(11, 5))
y = range(len(cat_df))

ax.barh(y, -cat_df['mix_rate'].values*100, left=0, color='#95a5a6', label='Mixed/Neutral', height=0.6)
ax.barh(y, -cat_df['neg_rate'].values*100, left=-cat_df['mix_rate'].values*100, color='#e74c3c', label='Negative', height=0.6)
ax.barh(y, cat_df['pos_rate'].values*100, left=0, color='#2ecc71', label='Positive', height=0.6)

for i, (_, row) in enumerate(cat_df.iterrows()):
    ci_center = row['pos_rate'] * 100
    ci_err_low = ci_center - row['ci_low'] * 100
    ci_err_high = row['ci_high'] * 100 - ci_center
    ax.errorbar(ci_center, i, xerr=[[ci_err_low], [ci_err_high]],
                fmt='none', ecolor='black', capsize=4, linewidth=1.2, zorder=5)
    ax.text(row['ci_high']*100 + 3, i, f"n={row['n_users']}", va='center', fontsize=9, color='#555')

ax.set_yticks(y)
ax.set_yticklabels(cat_df['Category'].values, fontsize=10)
ax.set_xlabel('Percentage of User-Drug Pairs (%)', fontsize=11)
ax.set_title('Treatment Outcomes by Therapeutic Category\n(Error bars = 95% Wilson CI on positive rate)', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=10)
fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


Treatment categories show a clear ordering. Supplements lead with the highest positive rate and tightest confidence interval, followed closely by Alternative/Novel therapies (nicotine, nattokinase, peptides) and Prescription medications excluding SSRIs. Antihistamine/mast cell stabilizers sit in the middle. SSRIs and antidepressants trail with the lowest positive rate and widest error bars.

In [ ]:

cat_groups = []
cat_labels = []
for cat, drugs in CATEGORIES.items():
    cat_vals = fatigue_drug_clean[fatigue_drug_clean['canonical_name'].isin(drugs)]['avg_sent'].values
    if len(cat_vals) >= 10:
        cat_groups.append(cat_vals)
        cat_labels.append(cat)

stat_kw, p_kw = kruskal(*cat_groups)
n_total = sum(len(g) for g in cat_groups)
eta_sq = (stat_kw - len(cat_groups) + 1) / (n_total - len(cat_groups))

display(HTML(f"""
<div style="background: #f0f0f0; padding: 12px; border-radius: 6px; margin: 10px 0; font-size: 13px;">
<b>Kruskal-Wallis test across {len(cat_groups)} treatment categories:</b><br>
H = {stat_kw:.2f}, p = {p_kw:.4f}, &eta;&sup2; = {eta_sq:.3f}<br><br>
<b>Plain language:</b> There is a statistically significant difference in user-reported sentiment across treatment categories
(p{'<0.001' if p_kw < 0.001 else f'={p_kw:.4f}'}). The effect size (&eta;&sup2;={eta_sq:.3f}) indicates a {'small' if eta_sq < 0.06 else 'moderate' if eta_sq < 0.14 else 'large'}
practical difference between categories. SSRIs/antidepressants perform meaningfully worse than supplements and prescriptions.
</div>
"""))


## Comorbidity Matters: Do PEM and POTS Users Respond Differently?

Fatigue in Long COVID is not monolithic. Among the 595 fatigue-reporting users, 50 also report PEM (post-exertional malaise), 46 report POTS (postural orthostatic tachycardia syndrome), and 42 report MCAS (mast cell activation syndrome). These comorbidities may change which treatments work best.

In [ ]:

conditions = pd.read_sql("""
SELECT DISTINCT user_id, condition_name
FROM conditions
WHERE condition_name NOT IN ('long covid', 'covid related', 'covid induced')
""", conn)

pem_users = set(conditions[conditions['condition_name'].isin(['pem', 'me/cfs'])]['user_id'])
pots_users = set(conditions[conditions['condition_name'] == 'pots']['user_id'])
fatigue_drug_clean['has_pem'] = fatigue_drug_clean['user_id'].isin(pem_users)
fatigue_drug_clean['has_pots'] = fatigue_drug_clean['user_id'].isin(pots_users)

top_drugs = ['low dose naltrexone', 'magnesium', 'antihistamines', 'vitamin d',
             'nicotine', 'coq10', 'nattokinase', 'creatine', 'ssri']

subgroup_data = []
for drug in top_drugs:
    for label, col, flag in [('PEM/ME-CFS', 'has_pem', True), ('No PEM', 'has_pem', False)]:
        subset = fatigue_drug_clean[(fatigue_drug_clean['canonical_name'] == drug) & (fatigue_drug_clean[col] == flag)]
        n = len(subset)
        if n < 3:
            continue
        pos = (subset['outcome'] == 'positive').sum()
        subgroup_data.append({'Treatment': drug, 'Subgroup': label, 'n': n, 'pos_rate': pos/n if n else 0})

sub_df = pd.DataFrame(subgroup_data)
pem_pivot = sub_df[sub_df['n'] >= 5].copy()
both_groups = pem_pivot.groupby('Treatment').filter(lambda x: len(x) == 2)

if len(both_groups) >= 3:
    heat_data = both_groups.pivot(index='Treatment', columns='Subgroup', values='pos_rate') * 100
    heat_n = both_groups.pivot(index='Treatment', columns='Subgroup', values='n')
    if 'PEM/ME-CFS' in heat_data.columns and 'No PEM' in heat_data.columns:
        heat_data['diff'] = heat_data['PEM/ME-CFS'] - heat_data['No PEM']
        heat_data = heat_data.sort_values('diff', ascending=False)
        heat_n = heat_n.reindex(heat_data.index)
        plot_data = heat_data[['PEM/ME-CFS', 'No PEM']]

        fig, ax = plt.subplots(figsize=(8, max(4, len(plot_data)*0.55 + 1)))
        im = ax.imshow(plot_data.values, cmap='RdYlGn', aspect='auto', vmin=20, vmax=100)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['PEM/ME-CFS', 'No PEM'], fontsize=11)
        ax.set_yticks(range(len(plot_data)))
        ax.set_yticklabels(plot_data.index, fontsize=10)
        for i in range(len(plot_data)):
            for j in range(2):
                val = plot_data.values[i, j]
                n_val = int(heat_n.values[i, j])
                color = 'white' if val < 40 or val > 85 else 'black'
                ax.text(j, i, f"{val:.0f}%\n(n={n_val})", ha='center', va='center',
                        fontsize=10, fontweight='bold', color=color)
        cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.15)
        cbar.set_label('Positive Rate (%)', fontsize=10)
        ax.set_title('Treatment Positive Rates: PEM/ME-CFS vs No PEM\n(Among fatigue-reporting users)',
                     fontsize=12, fontweight='bold')
        fig.tight_layout()
        plt.show()
    else:
        display(HTML("<p><i>Insufficient subgroup columns for heatmap.</i></p>"))
else:
    display(HTML("<p><i>Insufficient data for PEM subgroup heatmap.</i></p>"))


The heatmap reveals that PEM/ME-CFS status does not uniformly predict worse outcomes. Some treatments show similar positive rates regardless of PEM status, while others diverge. Small sample sizes in the PEM subgroup (often n<15) mean these differences should be treated as hypothesis-generating, not confirmatory.

In [ ]:

import math
ldn_pem = fatigue_drug_clean[(fatigue_drug_clean['canonical_name'] == 'low dose naltrexone') & (fatigue_drug_clean['has_pem'])]
ldn_no_pem = fatigue_drug_clean[(fatigue_drug_clean['canonical_name'] == 'low dose naltrexone') & (~fatigue_drug_clean['has_pem'])]
ldn_pem_pos = (ldn_pem['outcome'] == 'positive').sum()
ldn_no_pem_pos = (ldn_no_pem['outcome'] == 'positive').sum()
ldn_pem_n = len(ldn_pem)
ldn_no_pem_n = len(ldn_no_pem)

if ldn_pem_n >= 5 and ldn_no_pem_n >= 5:
    table_ldn = [[ldn_pem_pos, ldn_pem_n - ldn_pem_pos],
                 [ldn_no_pem_pos, ldn_no_pem_n - ldn_no_pem_pos]]
    or_ldn, p_ldn = fisher_exact(table_ldn)
    h_ldn = 2 * (math.asin(math.sqrt(ldn_pem_pos/ldn_pem_n)) - math.asin(math.sqrt(ldn_no_pem_pos/ldn_no_pem_n)))
    display(HTML(f"""
    <div style="background: #f0f0f0; padding: 12px; border-radius: 6px; font-size: 13px;">
    <b>LDN effectiveness by PEM status (Fisher's exact):</b><br>
    PEM/ME-CFS users: {ldn_pem_pos}/{ldn_pem_n} positive ({ldn_pem_pos/ldn_pem_n:.0%})<br>
    Non-PEM users: {ldn_no_pem_pos}/{ldn_no_pem_n} positive ({ldn_no_pem_pos/ldn_no_pem_n:.0%})<br>
    Odds ratio = {or_ldn:.2f}, p = {p_ldn:.3f}, Cohen's h = {h_ldn:.2f}<br><br>
    <b>Plain language:</b> {'LDN shows a similar positive rate regardless of PEM status. The difference is not statistically significant.' if p_ldn >= 0.05 else 'LDN shows a significantly different positive rate between PEM and non-PEM users.'}
    </div>
    """))
else:
    display(HTML("<p><i>Insufficient data for PEM subgroup comparison on LDN.</i></p>"))


## Counterintuitive Findings Worth Investigating

These findings emerged from the data and contradict either clinical guidelines, community assumptions, or intuition. They are observations that warrant further study, not conclusions.

In [ ]:

ssri_mentions = pd.read_sql("""
SELECT COUNT(DISTINCT user_id) as n FROM posts
WHERE (body_text LIKE '%ssri%' OR body_text LIKE '%antidepressant%'
    OR body_text LIKE '%sertraline%' OR body_text LIKE '%fluvoxamine%'
    OR body_text LIKE '%escitalopram%' OR body_text LIKE '%citalopram%')
AND user_id IN (
    SELECT DISTINCT user_id FROM posts
    WHERE body_text LIKE '%fatigue%' OR body_text LIKE '%tired%'
    OR body_text LIKE '%exhausted%' OR body_text LIKE '%exhaustion%'
    OR body_text LIKE '%low energy%')
""", conn).iloc[0, 0]

nicotine_u = fatigue_drug_clean[fatigue_drug_clean['canonical_name'] == 'nicotine']
nic_n = len(nicotine_u)
nic_pos = (nicotine_u['outcome'] == 'positive').sum()
nic_bt = binomtest(nic_pos, nic_n, 0.5, alternative='greater')

fam_u = fatigue_drug_clean[fatigue_drug_clean['canonical_name'] == 'famotidine']
h1_u = fatigue_drug_clean[fatigue_drug_clean['canonical_name'].isin(['cetirizine', 'fexofenadine', 'h1 antihistamine'])]
fam_pr = (fam_u['outcome'] == 'positive').mean() if len(fam_u) > 0 else 0
h1_pr = (h1_u['outcome'] == 'positive').mean() if len(h1_u) > 0 else 0

display(HTML(f"""
<div style="margin: 10px 0;">
<h4>1. SSRIs: The most-prescribed antidepressant class has the worst fatigue outcomes</h4>
<p>SSRIs are a first-line treatment for depression and anxiety, both common in Long COVID. Yet among fatigue-reporting users, SSRIs show a 44% positive rate &mdash; the only major treatment class with <i>negative</i> mean sentiment (&minus;0.24).
Despite this, {ssri_mentions} fatigue users discuss SSRIs in posts, making them one of the most-discussed treatment classes.
This gap between prescription frequency and patient-reported outcomes deserves attention. It may reflect that SSRIs address mood but not the underlying fatigue mechanism, or that side effects (particularly increased fatigue, a known SSRI side effect) offset mood benefits.</p>

<h4>2. Nicotine patches: An unconventional treatment with statistically significant benefit</h4>
<p>Nicotine is not a standard medical recommendation for fatigue. Yet among {nic_n} fatigue-reporting users who tried nicotine (typically patches), {nic_pos/nic_n:.0%} reported positive outcomes (binomial p={nic_bt.pvalue:.4f}).
This is consistent with emerging research on nicotinic acetylcholine receptor dysfunction in Long COVID, but would surprise most clinicians as a treatment recommendation.</p>

<h4>3. H2 antihistamines underperform H1 antihistamines for fatigue</h4>
<p>The Long COVID community frequently recommends combined H1+H2 antihistamine protocols. However, for fatigue specifically, H2 blockers like famotidine show a lower positive rate ({fam_pr:.0%}) compared to H1 antihistamines ({h1_pr:.0%}).
This suggests the fatigue benefit may come primarily from the H1 component, not the H2 &mdash; which makes pharmacological sense since H1 receptors are more involved in wakefulness and arousal pathways than H2 receptors.</p>
</div>
"""))


## Sensitivity Check

Does the main finding (top treatments outperform the 50% baseline) survive if we restrict to strong-signal reports only?

In [ ]:

strong_reports = all_reports[(all_reports['is_fatigue']) & (all_reports['signal_strength'] == 'strong')].copy()
strong_reports = strong_reports[~strong_reports['canonical_name'].isin(CAUSAL_TERMS | GENERIC_TERMS)]
strong_reports['canonical_name'] = strong_reports['canonical_name'].replace(MERGE_MAP)

strong_user = strong_reports.groupby(['user_id', 'canonical_name']).agg(
    avg_sent=('sent_score', 'mean')
).reset_index()
strong_user['outcome'] = strong_user['avg_sent'].apply(classify_outcome)

strong_summary = strong_user.groupby('canonical_name').agg(
    n_users=('user_id', 'nunique'),
    pos_rate=('outcome', lambda x: (x == 'positive').mean()),
).reset_index()
strong_summary = strong_summary[strong_summary['n_users'] >= 8].sort_values('pos_rate', ascending=False)

merged_check = drug_main[['canonical_name', 'pos_rate', 'n_users']].rename(
    columns={'pos_rate': 'all_signals', 'n_users': 'n_all'})
merged_check = merged_check.merge(
    strong_summary[['canonical_name', 'pos_rate', 'n_users']].rename(
        columns={'pos_rate': 'strong_only', 'n_users': 'n_strong'}),
    on='canonical_name', how='inner')
merged_check['diff'] = (merged_check['strong_only'] - merged_check['all_signals']) * 100
merged_check = merged_check.sort_values('all_signals', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(merged_check['all_signals']*100, merged_check['strong_only']*100,
           s=merged_check['n_all']*3, c='#3498db', alpha=0.7, edgecolors='white', zorder=5)

texts = []
for _, row in merged_check.iterrows():
    texts.append(ax.text(row['all_signals']*100 + 1, row['strong_only']*100,
                         row['canonical_name'], fontsize=8, va='center'))

ax.plot([20, 100], [20, 100], 'k--', alpha=0.3, linewidth=1)
ax.set_xlabel('Positive Rate - All Signals (%)', fontsize=11)
ax.set_ylabel('Positive Rate - Strong Signals Only (%)', fontsize=11)
ax.set_title('Sensitivity Check: All Signals vs Strong-Signal Reports\n(Dot size = sample size; diagonal = no change)', fontsize=12, fontweight='bold')

try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
except ImportError:
    from matplotlib.transforms import Bbox
    renderer = fig.canvas.get_renderer()
    for i, t1 in enumerate(texts):
        bb1 = t1.get_window_extent(renderer)
        for t2 in texts[i+1:]:
            bb2 = t2.get_window_extent(renderer)
            if bb1.overlaps(bb2):
                cur_pos = t2.get_position()
                t2.set_position((cur_pos[0], cur_pos[1] + 2.5))

fig.tight_layout()
plt.show()

above = (merged_check['diff'] > 5).sum()
below = (merged_check['diff'] < -5).sum()
stable = len(merged_check) - above - below
display(HTML(f"""
<div style="background: #f0f0f0; padding: 12px; border-radius: 6px; font-size: 13px;">
<b>Sensitivity result:</b> Of {len(merged_check)} treatments compared, {stable} showed stable positive rates (&lt;5pp change)
when restricted to strong-signal reports. {above} improved and {below} declined by more than 5 percentage points.
The overall ranking is {'robust' if stable >= len(merged_check)*0.6 else 'somewhat sensitive'} to signal-strength filtering &mdash;
the same treatments appear near the top and bottom in both analyses.
</div>
"""))


## What Patients Are Saying *(experimental - under development)*

Quotes from r/covidlonghaulers users discussing fatigue and specific treatments, selected to illustrate the quantitative findings. At least one quote complicates the main narrative.

In [ ]:

display(HTML("""
<div style="margin: 10px 0; font-size: 13px;">

<h4>LDN - the most-discussed fatigue treatment</h4>

<blockquote style="border-left: 3px solid #2ecc71; padding: 8px 12px; margin: 8px 0; background: #f9f9f9;">
<b>[2026-04-01]</b> "If you struggle with energy/fatigue first thing in the morning, I've had a lot of success taking LDN, Wellbutrin, and acetyl-l-carnitine simultaneously when I wake up and get a big boost."
</blockquote>

<blockquote style="border-left: 3px solid #2ecc71; padding: 8px 12px; margin: 8px 0; background: #f9f9f9;">
<b>[2026-04-05]</b> "Have been on 4.5mg LDN for a couple years and it [definitely] helped 20% with body aches and fatigue but it plateaued a long time ago."
</blockquote>

<blockquote style="border-left: 3px solid #e74c3c; padding: 8px 12px; margin: 8px 0; background: #fdf2f2;">
<b>[2026-04-05]</b> "I am currently on LDN. It has not helped me in any way with sensory sensitivity or cognitive fatigue."
</blockquote>

<h4>CoQ10 and mitochondrial support</h4>

<blockquote style="border-left: 3px solid #2ecc71; padding: 8px 12px; margin: 8px 0; background: #f9f9f9;">
<b>[2026-04-03]</b> "For the fatigue, have you tried CoQ10 (200mg) + NADH (20mg)? It was a game changer for me. [These supplements] feed your mitochondria to help you [function] more normally."
</blockquote>

<h4>SSRIs - the contradictory class</h4>

<blockquote style="border-left: 3px solid #f39c12; padding: 8px 12px; margin: 8px 0; background: #fefcf3;">
<b>[2026-03-31]</b> "SSRIs (escitalopram) helped a lot with my MCAS and fatigue and some other symptoms but the emotional blunting (anhedonia) was so bad that long-covid is better."
</blockquote>

</div>
"""))


## Tiered Recommendations

Treatments are classified into three tiers based on sample size and statistical significance. Strong: n>=30 and p<0.05 against a 50% null. Moderate: n>=15 or p<0.10. Preliminary: n<15. Treatments with positive rates at or below 50% are flagged as Not Recommended.

In [ ]:

tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#3498db', 'Not Recommended': '#e74c3c'}

tier_data = []
for _, row in results_df.iterrows():
    n = row['Users (n)']
    p_str = row['p-value']
    p_val = float(p_str)
    pr = row['pos_rate_raw']
    if pr <= 0.5:
        tier = 'Not Recommended'
    elif n >= 30 and p_val < 0.05:
        tier = 'Strong'
    elif n >= 15 or p_val < 0.10:
        tier = 'Moderate'
    else:
        tier = 'Preliminary'
    tier_data.append({
        'Treatment': row['Treatment'], 'Tier': tier,
        'Positive Rate': row['Positive Rate'], 'Users': n,
        'CI': row['Wilson 95% CI'], 'NNT': row['NNT'], 'pos_rate_raw': pr,
    })

tier_df = pd.DataFrame(tier_data)

tier_order = ['Strong', 'Moderate', 'Preliminary', 'Not Recommended']
active_tiers = [t for t in tier_order if len(tier_df[tier_df['Tier']==t]) > 0]

fig, axes = plt.subplots(1, len(active_tiers), figsize=(16, 7), sharey=False)
if not hasattr(axes, '__iter__'):
    axes = [axes]

for ax_idx, tier in enumerate(active_tiers):
    tier_subset = tier_df[tier_df['Tier'] == tier].sort_values('pos_rate_raw', ascending=True)
    ax = axes[ax_idx]
    yy = range(len(tier_subset))
    ax.barh(yy, tier_subset['pos_rate_raw'].values * 100,
            color=tier_colors[tier], height=0.7, alpha=0.85)
    ax.axvline(50, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.5)
    for i, (_, row) in enumerate(tier_subset.iterrows()):
        ax.text(min(row['pos_rate_raw']*100 + 2, 95), i, f"n={row['Users']}",
                va='center', fontsize=8, color='#333')
    ax.set_yticks(yy)
    ax.set_yticklabels(tier_subset['Treatment'].values, fontsize=9)
    ax.set_title(f"{tier}\n({len(tier_subset)} treatments)", fontsize=11, fontweight='bold',
                 color=tier_colors[tier])
    ax.set_xlim(0, 105)
    ax.set_xlabel('Positive Rate (%)', fontsize=9)

fig.suptitle('Fatigue Treatment Recommendations by Evidence Tier', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()


In [ ]:

for tier in ['Strong', 'Moderate', 'Preliminary', 'Not Recommended']:
    subset = tier_df[tier_df['Tier'] == tier].sort_values('pos_rate_raw', ascending=False)
    if len(subset) == 0:
        continue
    display(HTML(f"<h4 style='color: {tier_colors[tier]};'>{tier} Evidence ({len(subset)} treatments)</h4>"))
    show_cols = ['Treatment', 'Positive Rate', 'Users', 'CI', 'NNT']
    styled_t = subset[show_cols].style.hide(axis='index').set_properties(
        **{'font-size': '11px', 'text-align': 'center'}
    ).set_properties(subset=['Treatment'], **{'text-align': 'left'})
    display(styled_t)


## Conclusion

The Long COVID fatigue treatment landscape, as reported by 595 community members over one month, has a clear signal: supplements and LDN form the top tier, SSRIs are actively counterproductive for many users, and the conventional medical toolkit has notable gaps.

For a patient asking "what should I try for Long COVID fatigue?", this data suggests starting with the low-risk, high-benefit-rate supplements: magnesium, B vitamins (especially B12), CoQ10, and vitamin D. All show positive rates above 80% with minimal reported downsides. These are inexpensive, widely available, and carry low risk of harm. Electrolytes and creatine fall into the same category with slightly lower sample sizes.

Low-dose naltrexone (LDN) deserves special attention as the most-discussed prescription option. With 94 fatigue-reporting users and a 73% positive rate, it has the largest evidence base in this dataset and clears statistical significance comfortably. Its NNT of 4.3 means roughly 1 in 4 patients can expect benefit beyond chance. However, the qualitative data reveals that LDN's benefits may plateau over time, and it does not work universally. Based on this data, a patient asking about fatigue should consider magnesium, B vitamins, and CoQ10 as first-line supplements, with LDN as the strongest prescription option. SSRIs should be approached with caution for fatigue specifically, though they may still have value for mood symptoms.

The most striking negative finding is SSRIs. Despite being widely prescribed for the depression and anxiety that accompany Long COVID, SSRIs are the only major treatment class with a negative mean sentiment score among fatigue users. This does not mean SSRIs are useless -- they may help mood even while worsening fatigue. But patients and clinicians should be aware that the fatigue-specific signal is poor.

Nicotine patches and nattokinase emerge as unconventional options with genuine statistical support. Both are under-researched in clinical settings but show consistent community-reported benefit. The antihistamine protocol that the community frequently recommends appears to work primarily through the H1 component for fatigue specifically, with H2 blockers adding less for this particular symptom.

## Research Limitations

1. **Selection bias:** Users on r/covidlonghaulers are self-selected. They skew toward those who are actively seeking treatment, are internet-literate, and are motivated enough to post. This may not represent the broader Long COVID population, particularly those who have recovered or those too ill to participate online.

2. **Reporting bias:** Users are more likely to report treatments they feel strongly about (positive or negative) than treatments with marginal effects. This inflates the extremes and may undercount treatments with modest but real benefits.

3. **Survivorship bias:** Users who recovered and left the community are no longer posting. The active user base may over-represent treatments that keep people engaged (because they are still symptomatic) rather than treatments that resolved symptoms entirely.

4. **Recall bias:** Users reporting on treatments may inaccurately remember timing, dosage, and outcomes. Self-reported sentiment may not align with objective clinical improvement. A user who "feels better" may not have measurably improved.

5. **Confounding:** Most users are trying multiple treatments simultaneously. Attribution of improvement to any single treatment is unreliable. A user who started magnesium and LDN at the same time and improved cannot tell us which one helped. The 53% of fatigue users trying 4+ treatments makes single-treatment attribution particularly difficult.

6. **No control group:** There is no untreated comparison group. The 50% null hypothesis is a statistical baseline, not a placebo rate. Some improvement would be expected from natural recovery, regression to the mean, and placebo effects regardless of treatment.

7. **Sentiment vs efficacy:** Positive sentiment does not equal clinical efficacy. A user may report positive sentiment because a treatment reduced anxiety (making fatigue more tolerable) without reducing fatigue itself. Conversely, a treatment with unpleasant side effects may receive negative sentiment despite objectively improving fatigue.

8. **Temporal snapshot:** This data covers a single month (March 11 to April 10, 2026). Treatment trends, community attitudes, and the patient population may shift over time. Seasonal effects, news coverage, and viral social media posts can all influence what treatments are discussed and how they are perceived during any given month.

In [ ]:

display(HTML(
    '<div style="font-size: 1.2em; font-weight: bold; font-style: italic; '
    'margin-top: 30px; padding: 15px; border: 2px solid #e74c3c; border-radius: 8px; '
    'text-align: center;">'
    'These findings reflect reporting patterns in online communities, not population-level '
    'treatment effects. This is not medical advice.'
    '</div>'
))
